# Student Performance — Intelex vs CuPy benchmark

Dataset: https://www.kaggle.com/datasets/amrmaree/student-performance-prediction

Models: XGBoost, Logistic Regression, Random Forest, AdaBoost, MLP.

Sections are grouped by implementation because `patch_sklearn()` patches the whole kernel: run **BareBones** and **CuPy** first, then the single patch cell, then **Intelex**.

In [ ]:
import warnings
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import cupy as cp
from sklearnex import patch_sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier


## BareBones (plain scikit-learn)

### XGBoost

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainBareBones():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Optimize Data Types
    data = optimize_datatypes(data)
    
    # Encode categorical features
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    data[categorical_columns] = data[categorical_columns].apply(lambda col: col.cat.codes)
    
    # Split the Data
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1).values  # Extract values for efficient processing
    y = data['Pass_Fail'].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model using XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)
    
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for speed
        'device': 'cuda',       # GPU acceleration
        'random_state': 42
    }
    model = xgb.train(params, dtrain, num_boost_round=100)
    
    # Evaluate the Model
    y_pred = (model.predict(dtest) > 0.5).astype(int)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (Optimized Barebones): {accuracy * 100:.2f}%")
    
    # Print some predictions
    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainBareBones()


Accuracy (Optimized Barebones): 100.00%
   Actual  Predicted
0       0          0
1       1          1
2       1          1
3       1          1
4       1          1


### Logistic Regression

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainBareBones():
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Define the categorical columns
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    
    # Optimize data types
    data = optimize_datatypes(data)
    
    # Encode categorical data
    data[categorical_columns] = data[categorical_columns].apply(lambda col: col.cat.codes)
    
    # Split the Data
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1)
    y = data['Pass_Fail']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    # Evaluate the Model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    
    # Print some predictions
    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainBareBones()


Accuracy: 100.00%
     Actual  Predicted
120       0          0
247       1          1
324       1          1
204       1          1
603       1          1


### Random Forest

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainBareBones():
    # Load the Data
    file_path = "student_performance_dataset.csv"  # Replace with your dataset's path
    print("Loading the dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Optimize data types
    print("\nOptimizing data types...")
    data = optimize_datatypes(data)

    # Preprocess the Data
    print("\nPreprocessing the data...")
    label_encoder = LabelEncoder()
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = label_encoder.fit_transform(data[col])

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1).values
    y = data['Pass_Fail'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the Data
    print("\nSplitting the data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the Model
    print("\nTraining the Random Forest Classifier...")
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Evaluate the Model
    print("\nEvaluating the model...")
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")

    # Print some predictions
    print("\nDisplaying some predictions...")
    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainBareBones()

Loading the dataset...
Dataset loaded successfully! Shape: (708, 10)

Optimizing data types...

Preprocessing the data...

Separating features and target variable...
Features shape: (708, 8), Target shape: (708,)

Splitting the data into training and testing sets...
Training set shape: (566, 8), Testing set shape: (142, 8)

Training the Random Forest Classifier...
Model training completed!

Evaluating the model...
Accuracy: 100.00%

Displaying some predictions...
   Actual  Predicted
0       0          0
1       1          1
2       1          1
3       1          1
4       1          1


### AdaBoost

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainBareBones():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Optimize Data Types
    data = optimize_datatypes(data)
    
    # Encode categorical features
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    encoder = LabelEncoder()
    for col in categorical_columns:
        if col in data.columns:
            data[col] = encoder.fit_transform(data[col])
    
    # Split the Data
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1).values
    y = data['Pass_Fail'].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model using AdaBoost
    model = AdaBoostClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate the Model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (AdaBoost): {accuracy * 100:.2f}%")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=1))

if __name__ == "__main__":
    mainBareBones()


Accuracy (AdaBoost): 100.00%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        71
           1       1.00      1.00      1.00        71

    accuracy                           1.00       142
   macro avg       1.00      1.00      1.00       142
weighted avg       1.00      1.00      1.00       142



### MLP

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainBareBones():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Optimize Data Types
    data = optimize_datatypes(data)
    
    # Encode categorical features
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    data[categorical_columns] = data[categorical_columns].apply(lambda col: col.cat.codes)
    
    # Split the Data
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1).values  # Extract values for efficient processing
    y = data['Pass_Fail'].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model using a Deep Neural Network
    model = MLPClassifier(
    hidden_layer_sizes=(50, 30),  
    max_iter=1200,               
    learning_rate='adaptive',            
    early_stopping=True,        
    random_state=42              
    )
    model.fit(X_train, y_train)
    
    # Evaluate the Model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    
    # Print some predictions
    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainBareBones()


Accuracy: 63.38%
   Actual  Predicted
0       0          0
1       1          1
2       1          1
3       1          1
4       1          1


## CuPy

### XGBoost

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 
                           'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category').cat.codes  # Encode as category

    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainCuPY():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Optimize datatypes
    data = optimize_datatypes(data)
    
    # Convert DataFrame to cuPy array
    X = cp.asarray(data.drop(['Student_ID', 'Pass_Fail'], axis=1).values, dtype=cp.float32)
    y = cp.asarray(data['Pass_Fail'].values, dtype=cp.int32)
    
    # Split the Data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model using XGBoost with GPU support
    dtrain = xgb.DMatrix(cp.asnumpy(X_train), label=cp.asnumpy(y_train))
    dtest = xgb.DMatrix(cp.asnumpy(X_test), label=cp.asnumpy(y_test))
    
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for speed
        'device': 'cuda',       # GPU acceleration
        'random_state': 42
    }
    model = xgb.train(params, dtrain, num_boost_round=100)
    
    # Evaluate the Model
    y_pred = (model.predict(dtest) > 0.5).astype(int)
    accuracy = accuracy_score(cp.asnumpy(y_test), y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    
    # Print some predictions
    predictions_df = pd.DataFrame({'Actual': cp.asnumpy(y_test), 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainCuPY()


Accuracy: 100.00%
   Actual  Predicted
0       0          0
1       1          1
2       1          1
3       1          1
4       1          1


### Logistic Regression

In [ ]:
def optimize_dtypes(df):
    # Optimize numeric columns
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif df[col].dtype == 'int64':
            df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

def maincuPY():
    # Load the Data
    print("Loading dataset...")
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Preprocess the Data (Optimize by converting to appropriate types first)
    print("Preprocessing data...")
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 
                           'Extracurricular_Activities', 'Pass_Fail']
    
    for col in categorical_columns:
        # Convert to 'category' dtype and encode
        data[col] = data[col].astype('category').cat.codes

    # Optimize numeric columns for memory efficiency
    data = optimize_dtypes(data)
    
    # Convert data to CuPy arrays
    print("Converting data to CuPy arrays...")
    X = cp.array(data.drop(['Student_ID', 'Pass_Fail'], axis=1).values, dtype=cp.float32)
    y = cp.array(data['Pass_Fail'].values, dtype=cp.int32)
    
    # Split the Data
    print("Splitting dataset...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model
    print("Training the model...")
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(cp.asnumpy(X_train), cp.asnumpy(y_train))  # Convert CuPy arrays to NumPy
    
    # Evaluate the Model
    print("Evaluating the model...")
    y_pred = model.predict(cp.asnumpy(X_test))  
    accuracy = accuracy_score(cp.asnumpy(y_test), y_pred)  
    print(f"Accuracy: {accuracy * 100:.2f}%")
    
    # Print sample predictions
    print("Sample predictions:")
    predictions_df = pd.DataFrame({'Actual': cp.asnumpy(y_test), 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    maincuPY()

Loading dataset...
Preprocessing data...
Converting data to CuPy arrays...
Splitting dataset...
Training the model...
Evaluating the model...
Accuracy: 100.00%
Sample predictions:
   Actual  Predicted
0       0          0
1       1          1
2       1          1
3       1          1
4       1          1


### Random Forest

In [ ]:
def optimize_datatypes(data):
    # Downcast integer and float columns to minimize memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainCuPY():
    # Load the Data
    print("Loading dataset...")
    try:
        data = pd.read_csv("student_performance_dataset.csv")
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return

    # Optimize datatypes for memory efficiency
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Preprocess the Data
    print("\nPreprocessing the data...")
    label_encoder = LabelEncoder()
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = label_encoder.fit_transform(data[col])

    # Convert DataFrame to CuPy arrays
    print("\nConverting data to CuPy arrays...")
    X = cp.asarray(data.drop(['Student_ID', 'Pass_Fail'], axis=1).to_numpy(dtype='float32'))
    y = cp.asarray(data['Pass_Fail'].to_numpy(dtype='int32'))

    # Split the Data
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the Model using RandomForest
    print("\nTraining the Random Forest Classifier...")
    model = RandomForestClassifier(random_state=42, n_estimators=100)
    model.fit(cp.asnumpy(X_train), cp.asnumpy(y_train))
    print("Model training completed!")

    # Evaluate the Model
    print("\nEvaluating the model...")
    y_pred = model.predict(cp.asnumpy(X_test))
    accuracy = accuracy_score(cp.asnumpy(y_test), y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")

    # Print some predictions
    print("\nDisplaying some predictions...")
    predictions_df = pd.DataFrame({'Actual': cp.asnumpy(y_test), 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainCuPY()

Loading dataset...
Dataset loaded successfully! Shape: (708, 10)

Optimizing datatypes...

Preprocessing the data...

Converting data to CuPy arrays...

Splitting data into training and testing sets...
Training set shape: (566, 8), Testing set shape: (142, 8)

Training the Random Forest Classifier...
Model training completed!

Evaluating the model...
Accuracy: 100.00%

Displaying some predictions...
   Actual  Predicted
0       0          0
1       1          1
2       1          1
3       1          1
4       1          1


### AdaBoost

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        cupy_array = cp.array(data[col], dtype=cp.int32)  # Convert to CuPy array
        data[col] = cupy_array.get()  # Convert back to NumPy
    for col in data.select_dtypes(include='float'):
        cupy_array = cp.array(data[col], dtype=cp.float32)  # Convert to CuPy array
        data[col] = cupy_array.get() 
    return data

def mainCuPY():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Optimize Data Types
    data = optimize_datatypes(data)
    
    # Encode categorical features
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    encoder = LabelEncoder()
    for col in categorical_columns:
        if col in data.columns:
            data[col] = encoder.fit_transform(data[col])
    
    # Split the Data
    X = cp.array(data.drop(['Student_ID', 'Pass_Fail'], axis=1).values)  # Convert features to CuPy array
    y = cp.array(data['Pass_Fail'].values)  # Convert target to CuPy array
    

    X_train, X_test, y_train, y_test = train_test_split(cp.asnumpy(X), cp.asnumpy(y), test_size=0.2, random_state=42)
    
    # Train the Model using AdaBoost
    model = AdaBoostClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate the Model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (AdaBoost): {accuracy * 100:.2f}%")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=1))

if __name__ == "__main__":
    mainCuPY()


Accuracy (AdaBoost): 100.00%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        71
           1       1.00      1.00      1.00        71

    accuracy                           1.00       142
   macro avg       1.00      1.00      1.00       142
weighted avg       1.00      1.00      1.00       142



### MLP

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Convert numerical columns to CuPy arrays
    for col in data.select_dtypes(include='int'):
        data[col] = cp.array(data[col], dtype=cp.int32).get()
    for col in data.select_dtypes(include='float'):
        data[col] = cp.array(data[col], dtype=cp.float32).get()
    
    return data

def mainCuPY():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Optimize Data Types
    data = optimize_datatypes(data)
    
    # Encode categorical features
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    data[categorical_columns] = data[categorical_columns].apply(lambda col: col.cat.codes)
    
    # Convert to CuPy arrays
    X = cp.array(data.drop(['Student_ID', 'Pass_Fail'], axis=1).values)  # Extract values for efficient processing
    y = cp.array(data['Pass_Fail'].values)
    
    # Split the Data
    X_train, X_test, y_train, y_test = train_test_split(X.get(), y.get(), test_size=0.2, random_state=42)
    
    # Train the Model using a Deep Neural Network
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    
    # Evaluate the Model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    
    # Print some predictions
    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainCuPY()


Accuracy: 63.38%
   Actual  Predicted
0       0          0
1       1          1
2       1          1
3       1          1
4       1          1


## Intelex (scikit-learn-intelex)

Apply the Intel patch once; everything below runs patched.

In [ ]:
patch_sklearn()  # Apply Intel optimizations


Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


### XGBoost

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainIntelex():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Optimize Data Types
    data = optimize_datatypes(data)
    
    # Encode categorical features
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    data[categorical_columns] = data[categorical_columns].apply(lambda col: col.cat.codes)
    
    # Split the Data
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1).values  # Extract values for efficient processing
    y = data['Pass_Fail'].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model using XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)
    
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for speed
        'device': 'cuda',       # GPU acceleration
        'random_state': 42
    }
    model = xgb.train(params, dtrain, num_boost_round=100)
    
    # Evaluate the Model
    y_pred = (model.predict(dtest) > 0.5).astype(int)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (Optimized Barebones): {accuracy * 100:.2f}%")
    
    # Print some predictions
    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainIntelex()

Accuracy (Optimized Barebones): 100.00%
   Actual  Predicted
0       0          0
1       1          1
2       1          1
3       1          1
4       1          1


### Logistic Regression

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainIntelex():
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Define the categorical columns
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    
    # Optimize data types
    data = optimize_datatypes(data)
    
    # Encode categorical data
    data[categorical_columns] = data[categorical_columns].apply(lambda col: col.cat.codes)
    
    # Split the Data
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1)
    y = data['Pass_Fail']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    # Evaluate the Model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")

if __name__ == "__main__":
    mainIntelex()


Accuracy: 100.00%


### Random Forest

In [ ]:
def optimize_datatypes(data):
    # Downcast integer and float columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainIntelex():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    print("Loading dataset...")
    try:
        data = pd.read_csv("student_performance_dataset.csv")
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return

    # Optimize datatypes for memory efficiency
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Preprocess the Data
    print("\nPreprocessing the data...")
    label_encoder = LabelEncoder()
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = label_encoder.fit_transform(data[col])

    # Split the Data
    print("\nSeparating features and target variable...")
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1)
    y = data['Pass_Fail']

    print("\nSplitting the data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the Model
    print("\nTraining the Random Forest Classifier...")
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Evaluate the Model
    print("\nEvaluating the model...")
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")

    # Print some predictions
    print("\nDisplaying some predictions...")
    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainIntelex()

Loading dataset...
Dataset loaded successfully! Shape: (708, 10)

Optimizing datatypes...

Preprocessing the data...

Separating features and target variable...

Splitting the data into training and testing sets...
Training set shape: (566, 8), Testing set shape: (142, 8)

Training the Random Forest Classifier...
Model training completed!

Evaluating the model...
Accuracy: 100.00%

Displaying some predictions...
     Actual  Predicted
120       0          0
247       1          1
324       1          1
204       1          1
603       1          1


### AdaBoost

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainIntelex():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Optimize Data Types
    data = optimize_datatypes(data)
    
    # Encode categorical features
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    encoder = LabelEncoder()
    for col in categorical_columns:
        if col in data.columns:
            data[col] = encoder.fit_transform(data[col])
    
    # Split the Data
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1).values
    y = data['Pass_Fail'].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model using AdaBoost
    model = AdaBoostClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate the Model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (AdaBoost): {accuracy * 100:.2f}%")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=1))

if __name__ == "__main__":
    mainIntelex()


Accuracy (AdaBoost): 100.00%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        71
           1       1.00      1.00      1.00        71

    accuracy                           1.00       142
   macro avg       1.00      1.00      1.00       142
weighted avg       1.00      1.00      1.00       142



### MLP

In [ ]:
def optimize_datatypes(data):
    # Convert categorical features to category dtype
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    for col in categorical_columns:
        data[col] = data[col].astype('category')
    
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainIntelex():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the Data
    data = pd.read_csv("student_performance_dataset.csv")
    
    # Optimize Data Types
    data = optimize_datatypes(data)
    
    # Encode categorical features
    categorical_columns = ['Gender', 'Parental_Education_Level', 'Internet_Access_at_Home', 'Extracurricular_Activities', 'Pass_Fail']
    data[categorical_columns] = data[categorical_columns].apply(lambda col: col.cat.codes)
    
    # Split the Data
    X = data.drop(['Student_ID', 'Pass_Fail'], axis=1).values  # Extract values for efficient processing
    y = data['Pass_Fail'].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the Model using a Deep Neural Network
    model = MLPClassifier(
    hidden_layer_sizes=(50, 30),  
    max_iter=1200,               
    learning_rate='adaptive',            
    early_stopping=True,        
    random_state=42              
    )
    model.fit(X_train, y_train)
    
    # Evaluate the Model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    
    # Print some predictions
    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    print(predictions_df.head())

if __name__ == "__main__":
    mainIntelex()


Accuracy: 63.38%
   Actual  Predicted
0       0          0
1       1          1
2       1          1
3       1          1
4       1          1
